# **Hoja de trabajo #1 - Aprendizaje por Refuerzo**

- Paula Barillas - 22764
- Gerardo Pineda - 22880
- Monica Salvatierra - 22249
- Bianca Calderón - 22272

Link del repositorio: https://github.com/paulabaal12/HDT1-RL 

---

## **Task 1**
---

> Responda las siguientes preguntas de forma clara, concisa y argumentada. No basta con definir: se espera que usted conecte el concepto con sus implicaciones prácticas o estratégicas.

---

##### 1. Elija un sistema real de su área de interés (puede ser de banca, telecomunicaciones, salud, videojuegos, robótica u otro). Identifique y justifique ara ese sistema: (a) qué representa el estado 𝑆𝑡, (b) qué acciones componen el espacio 𝒜, (c) cómo diseñaría la función de recompensa 𝑅, y (d) si el entorno es totalmente observable o parcialmente observable, y por qué. 

En este caso escogimos un robot aspirador, ya que estas se mueven por un cuarto dividido en casillas, ya que tiene que limpiar y luego volver a su base a cargar. En este caso el robot esta tomando decisiones secuenciales en a donde tiene que moverse y el premio que en este caso es terminar de limpiar y cargarse donde llega varios pasos después de esas decisiones.  

**(a) Estado $S_t$**
Representa la casilla/cuadro donde está el robot dentro del cuarto; básicamente es su **posición en la cuadrícula**.

**(b) Espacio de acciones $\mathcal{A}$**
Es **discreto**, ya que se compone de

$$\mathcal{A} = \{\text{arriba},\ \text{abajo},\ \text{izquierda},\ \text{derecha}\}$$

**(c) Función de recompensa $R$**
Como la recompensa es una **señal de diseño** y no un dato observado, la definimos así:

| Situación | Recompensa |
|---|---|
| Limpia una casilla sucia | $+1$ |
| Cada paso que da (gasta batería) | $-1$ |
| Llega a su base a cargarse | $+10$ |

El $-1$ por paso lo empuja a ser eficiente y a no dar vueltas de más, respetando la hipótesis de recompensa, donde el objetivo es limpiar rápido y no quedarse sin batería.

**(d) Observabilidad**
Es totalmente observable , ya que el robot conoce la distribución/mapa del cuarto y su propia posición, por lo cual la observación coincide con el estado real

$$O_t = S_t$$

---

##### 2. Para el sistema que eligió, argumente qué valor de 𝛾 sería más apropiado y por qué. Incluya al menos un ejemplo numérico que ilustre las consecuencias de elegir 𝛾 alto versus 𝛾 bajo para ese dominio específico. 

Para el robot aspirador conviene un **$\gamma$ alto (entre 0.9 y 0.99)**, ya que la recompensa más importante ($+10$ por llegar a la base) se obtiene varios pasos después de las decisiones que la hicieron posible. Un $\gamma$ bajo diluiría demasiado esa señal, generando un comportamiento **miope** que prioriza lo inmediato sin pensar en recargar a tiempo.

**Ejemplo numérico** el robot está a 5 pasos de la base y tiene una casilla sucia justo al lado ($+1$ inmediato):

| $\gamma$ | Valor descontado de la base | Decisión del robot |
|---|---|---|
| $0.9$ | $0.9^5 \times 10 \approx 5.9$ | Prioriza ir a cargar  (5.9 > 1) |
| $0.5$ | $0.5^5 \times 10 \approx 0.31$ | Prefiere seguir limpiando  (0.31 < 1) |

Con $\gamma = 0.5$ el robot arriesga quedarse sin batería en medio del cuarto, mientras que con $\gamma = 0.9$ toma la decisión correcta de recargar a tiempo.

---

##### 3. Describa en lenguaje natural una política determinista y una política estocástica para su sistema elegido. ¿En qué escenario preferiría una política estocástica sobre una determinista?  

Una política determinista asigna siempre la misma acción a cada estado. Por ejemplo, si el robot está en la casilla (3,2) siempre se moverá hacia arriba, generando una ruta fija y predecible como un patrón serpenteante para cubrir el cuarto. Una política estocástica, en cambio, define una distribución de probabilidad sobre las acciones para cada estado (por ejemplo, 70% de probabilidad de moverse arriba y 30% a la derecha), por lo que el robot puede tomar rutas distintas en situaciones similares. 

Preferiríamos una política estocástica cuando el entorno tiene incertidumbre o puede cambiar, como muebles que se mueven o personas caminando por el cuarto, ya que el robot necesita variar su ruta para no quedarse atorado en un patrón fijo. También es útil durante el entrenamiento para explorar distintas opciones y evitar quedarse en una solución subóptima, o si hay varios robots limpiando el mismo espacio y conviene que no sigan siempre la misma ruta para evitar choques. 

## **Task 2**
---

In [1]:
"""
Entorno: GridWorld 4x4 simple
- El agente parte de (0,0) y busca llegar a (3,3)
- Recompensa: -1 por cada paso, +10 al llegar a la meta, -5 si cae en celda trampa
- El agente sigue una política aleatoria (no aprende todavía — solo observamos el ciclo)
"""

import numpy as np
import random

# Constantes
GRID_SIZE = 4
GOAL_STATE = (3, 3)          # Estado terminal con recompensa positiva
TRAP_STATES = {(1, 1), (2, 3)}  # Estados trampa
MAX_STEPS = 50               # Horizonte máximo del episodio
GAMMA = 0.95                 # Factor de descuento

# Acciones
# Espacio de acciones discreto: 4 direcciones posibles
ACTIONS = {
    0: (-1, 0),   # Arriba
    1: (1, 0),    # Abajo
    2: (0, -1),   # Izquierda
    3: (0, 1),    # Derecha
}
ACTION_NAMES = {0: "↑", 1: "↓", 2: "←", 3: "→"}

# Entorno
class GridWorldEnv:
    """
    Representa el entorno (Environment) en el vocabulario de RL.
    
    Responsabilidades:
    - Mantener el estado actual del mundo
    - Transicionar al nuevo estado dado una acción
    - Emitir la recompensa R_{t+1} después de cada transición
    """
    
    def reset(self):
        """Inicializa el entorno y devuelve el estado inicial S_0."""
        self.state = (0, 0)
        self.done = False
        return self.state
    
    def step(self, action):
        """
        Ejecuta un paso del ciclo agente-entorno.
        
        Recibe: A_t (action)
        Devuelve: S_{t+1}, R_{t+1}, done
        
        Esta función implementa la función de transición P(S_{t+1} | S_t, A_t)
        y la función de recompensa R(S_t, A_t, S_{t+1}).
        """
        if self.done:
            raise RuntimeError("El episodio ya terminó. Llama reset().")
        
        dr, dc = ACTIONS[action]
        r, c = self.state
        
        # La transición respeta los bordes del grid (entorno determinista)
        new_r = max(0, min(GRID_SIZE - 1, r + dr))
        new_c = max(0, min(GRID_SIZE - 1, c + dc))
        next_state = (new_r, new_c)
        
        # Función de recompensa: señal de diseño del ingeniero
        if next_state == GOAL_STATE:
            reward = +10.0    # Meta alcanzada
            self.done = True
        elif next_state in TRAP_STATES:
            reward = -5.0     # Trampa activada
            self.done = True
        else:
            reward = -1.0     # Costo por paso (incentiva llegar rápido)
        
        self.state = next_state
        return next_state, reward, self.done


# Agente con politica aleatoria
class RandomAgent:
    """
    Agente con política estocástica uniforme.
    
    Política π(a|s) = 1/|A| para todo a — distribución uniforme.
    No aprende — sirve para ilustrar el ciclo de interacción.
    """
    
    def __init__(self, action_space_size):
        self.n_actions = action_space_size
    
    def select_action(self, state):
        """
        Muestrea una acción de la política estocástica uniforme.
        
        En un agente que aprende (Q-learning, PPO, etc.), esta función
        consultaría una tabla Q o una red neuronal.
        """
        return random.randint(0, self.n_actions - 1)


# Ciclo de agnte entorno
def run_episode(env, agent, gamma=GAMMA, max_steps=MAX_STEPS, verbose=True):
    """
    Ejecuta un episodio completo y calcula el retorno G_0.
    
    Implementa directamente el ciclo de la Diapositiva 3:
        S_0 → A_0 → R_1, S_1 → A_1 → R_2, S_2 → ...
    """
    state = env.reset()                    # S_0
    trajectory = []                        # Almacena (S_t, A_t, R_{t+1})
    total_reward = 0.0
    
    if verbose:
        print(f"\n{'═'*50}")
        print(f"INICIO DEL EPISODIO  Estado inicial: {state}")
        print(f"{'─'*50}")
    
    for t in range(max_steps):
        action = agent.select_action(state)                    # A_t ~ π(·|S_t)
        next_state, reward, done = env.step(action)            # Transición
        
        trajectory.append((state, action, reward, next_state))
        total_reward += reward
        
        if verbose:
            status = "✓ META" if next_state == GOAL_STATE \
                     else "✗ TRAMPA" if next_state in TRAP_STATES \
                     else ""
            print(f"  t={t:2d}  S={state}  A={ACTION_NAMES[action]}  "
                  f"R={reward:+.1f}  S'={next_state}  {status}")
        
        state = next_state
        
        if done:
            break
    
    # Calcula el retorno descontado G_0 = Σ γ^k * R_{k+1}
    # Procesa la trayectoria en reversa para acumulación eficiente
    G = 0.0
    returns = []
    for _, _, r, _ in reversed(trajectory):
        G = r + gamma * G                  # G_t = R_{t+1} + γ * G_{t+1}
        returns.insert(0, G)
    
    if verbose:
        print(f"{'─'*50}")
        print(f"  Pasos totales        : {len(trajectory)}")
        print(f"  Recompensa acumulada : {total_reward:+.1f}")
        print(f"  Retorno G_0 (γ={gamma}): {returns[0]:.4f}")
        print(f"{'═'*50}\n")
    
    return trajectory, returns


# Experimentar con multiples episodios
def run_experiment(n_episodes=5):
    """
    Corre múltiples episodios para ilustrar la variabilidad de una política
    estocástica — con la misma política, los resultados varían por azar.
    """
    env = GridWorldEnv()
    agent = RandomAgent(action_space_size=len(ACTIONS))
    
    print("SIMULACIÓN: Ciclo Agente-Entorno en GridWorld 4×4")
    print("Política: aleatoria uniforme (sin aprendizaje)")
    print(f"Factor de descuento γ = {GAMMA}")
    
    all_returns = []
    for ep in range(n_episodes):
        print(f"\n▶ Episodio {ep + 1}/{n_episodes}")
        _, returns = run_episode(env, agent, verbose=True)
        all_returns.append(returns[0])
    
    print("\n" + "═"*50)
    print("RESUMEN DEL EXPERIMENTO")
    print(f"  Retorno promedio G_0 : {np.mean(all_returns):.4f}")
    print(f"  Retorno máximo       : {max(all_returns):.4f}")
    print(f"  Retorno mínimo       : {min(all_returns):.4f}")
    print("═"*50)
    print("\nObservación: la variabilidad entre episodios ilustra por qué")
    print("el objetivo de RL es el RETORNO ESPERADO, no un retorno fijo.")


if __name__ == "__main__":
    random.seed(42)
    np.random.seed(42)
    run_experiment(n_episodes=5)

SIMULACIÓN: Ciclo Agente-Entorno en GridWorld 4×4
Política: aleatoria uniforme (sin aprendizaje)
Factor de descuento γ = 0.95

▶ Episodio 1/5

══════════════════════════════════════════════════
INICIO DEL EPISODIO  Estado inicial: (0, 0)
──────────────────────────────────────────────────
  t= 0  S=(0, 0)  A=↑  R=-1.0  S'=(0, 0)  
  t= 1  S=(0, 0)  A=↑  R=-1.0  S'=(0, 0)  
  t= 2  S=(0, 0)  A=←  R=-1.0  S'=(0, 0)  
  t= 3  S=(0, 0)  A=↓  R=-1.0  S'=(1, 0)  
  t= 4  S=(1, 0)  A=↓  R=-1.0  S'=(2, 0)  
  t= 5  S=(2, 0)  A=↓  R=-1.0  S'=(3, 0)  
  t= 6  S=(3, 0)  A=↑  R=-1.0  S'=(2, 0)  
  t= 7  S=(2, 0)  A=↑  R=-1.0  S'=(1, 0)  
  t= 8  S=(1, 0)  A=→  R=-5.0  S'=(1, 1)  ✗ TRAMPA
──────────────────────────────────────────────────
  Pasos totales        : 9
  Recompensa acumulada : -13.0
  Retorno G_0 (γ=0.95): -10.0487
══════════════════════════════════════════════════


▶ Episodio 2/5

══════════════════════════════════════════════════
INICIO DEL EPISODIO  Estado inicial: (0, 0)
──────────

---

### **Prompt Utilizado**


### **Por que funciono este prompt**

Funcionó porque 

---